In [5]:
import pandas as pd
import networkx as nx
import pickle

# 1. Load the Graph using standard pickle
with open('../data/manhattan/manhattan_graph.gpickle', 'rb') as f:
    G = pickle.load(f)

# 2. Load the POI Data (pandas handles pickles fine)
poi_df = pd.read_pickle('../data/manhattan/manhattan_poi.pkl')

print(f"Graph loaded: {len(G.nodes)} nodes, {len(G.edges)} edges")
print(f"POI Data loaded: {len(poi_df)} rows")

C:\Users\adan\AppData\Local\Temp\ipykernel_5584\1187952574.py:7: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  G = pickle.load(f)
C:\Users\adan\AppData\Local\Temp\ipykernel_5584\1187952574.py:7: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  G = pickle.load(f)
c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\.venv\Lib\site-packages\pandas\compat\pickle_compat.py:80: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  stack[-1] = func(*args)
C:\Users\adan\miniconda3\Lib\pickle.py:1760: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may 

Graph loaded: 74137 nodes, 209248 edges
POI Data loaded: 20979 rows


In [6]:
# Cell: Identifying our Landmark Keys
print("--- POI COLUMNS ---")
print(poi_df.columns.tolist())

print("\n--- FIRST 5 NODE IDS ---")
# This tells us if they are strings, ints, or prefixed
print(list(G.nodes)[0:5])

print("\n--- SAMPLE LANDMARK DATA ---")
# Let's see what a typical row looks like (filtering for name if it exists)
if 'name' in poi_df.columns:
    print(poi_df[['name', 'osmid']].head(5))
else:
    print(poi_df.head(5))

--- POI COLUMNS ---
['unique_id', 'osmid', 'element_type', 'alt_name', 'ele', 'gnis:Class', 'gnis:County', 'gnis:County_num', 'gnis:ST_alpha', 'gnis:ST_num', 'gnis:id', 'import_uuid', 'is_in', 'name', 'name:azb', 'name:fa', 'name:ja', 'name:ko', 'name:ru', 'name:uk', 'name:zh', 'place', 'geometry', 'highway', 'ref', 'source', 'network', 'operator', 'public_transport', 'railway', 'railway:ref', 'train', 'created_by', 'railway:position', 'barrier', 'payment:cash', 'junction', 'old_ref', 'crossing', 'button_operated', 'tactile_paving', 'traffic_signals:sound', 'direction', 'stop', 'maxspeed', 'segregated', 'bus', 'bicycle', 'historic', 'man_made', 'surveillance:type', 'traffic_calming', 'cycleway', 'crossing:island', 'subway', 'wheelchair', 'fixme', 'note', 'name:etymology:wikidata', 'amenity', 'iata', 'brand', 'charge', 'fee', 'manufacturer', 'material', 'surveillance', 'toll', 'website', 'alt_name:pt', 'alt_name:vi', 'importance', 'is_in:continent', 'is_in:country', 'is_in:country_code'

In [7]:
# --- THE 1# SANITY CHECK ---

# 1. Get a valid sample from your data
# We filter for rows that actually have a name to make it readable
valid_samples = poi_df[poi_df['name'].notna()]

if valid_samples.empty:
    print("❌ No named landmarks found in poi_df. Checking by index instead...")
    sample_poi = poi_df.iloc[0]
    poi_name = "Unnamed Landmark"
else:
    sample_poi = valid_samples.iloc[0]
    poi_name = sample_poi['name']

# 2. Extract the OSMID (handling potential # prefixes in the data)
raw_osmid = str(sample_poi['osmid']).replace('#', '') 

# 3. Construct the ID strings
projected_id = f"1#{raw_osmid}"
simple_id = f"#{raw_osmid}"

print(f"🧪 Testing Landmark: '{poi_name}'")
print(f"🆔 Extracted OSMID: {raw_osmid}")
print("-" * 30)

# 4. Check existence in the Graph
found_projected = projected_id in G.nodes
found_simple = simple_id in G.nodes

print(f"Checking '1#{raw_osmid}': {'✅ SUCCESS' if found_projected else '❌ NOT FOUND'}")
print(f"Checking '#{raw_osmid}':  {'✅ SUCCESS' if found_simple else '❌ NOT FOUND'}")

# 5. Connectivity Check (Crucial for Navigation)
if found_projected:
    degree = G.degree(projected_id)
    print(f"🔗 Connectivity: This node has {degree} street connections.")
elif found_simple:
    degree = G.degree(simple_id)
    print(f"🔗 Connectivity: The simple node has {degree} street connections.")
else:
    print("⚠️ WARNING: Neither node format was found in the graph. We may need to check the first 5 Node IDs again.")

🧪 Testing Landmark: 'Hell's Kitchen'
🆔 Extracted OSMID: 666
------------------------------
Checking '1#666': ✅ SUCCESS
Checking '#666':  ✅ SUCCESS
🔗 Connectivity: This node has 12 street connections.
